# EEG 数据集探索与分析

本 Notebook 用于探索和可视化公开的 EEG 数据集：
- PhysioNet EEG Motor Movement/Imagery
- DEAP (情绪)
- SEED (情绪)

## 环境准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import sys

# 添加项目路径
sys.path.append('../backend')

from app.data_loaders import (
    PhysioNetLoader,
    DEAPLoader,
    SEEDLoader,
    create_dataset_manager
)
from app.eeg_processor_v2 import EEGProcessor, EEGConfig

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 1. PhysioNet 数据集探索

In [ ]:
# 配置数据路径
PHYSIONET_DIR = '/path/to/physionet'

# 创建加载器
loader = PhysioNetLoader(PHYSIONET_DIR)

# 获取数据集信息
info = loader.get_info()
print(f"数据集：{info['name']}")
print(f"被试数量：{info['n_subjects']}")
print(f"通道数：{info['n_channels']}")
print(f"采样率：{info['sample_rate']}Hz")
print(f"任务类型：{info['tasks']}")

In [ ]:
# 获取可用被试列表
subjects = loader.get_subjects()
print(f"可用被试：{len(subjects)} 个")
print(f"前 10 个被试：{subjects[:10]}")

In [ ]:
# 加载第一个被试的数据
if subjects:
    subject_id = subjects[0]
    eeg_data, metadata = loader.load_subject(subject_id)
    
    print(f"\n被试 {subject_id} 数据:")
    print(f"数据形状：{eeg_data.shape}")
    print(f"通道数：{eeg_data.shape[0]}")
    print(f"样本数：{eeg_data.shape[1]}")
    print(f"时长：{metadata['duration']:.1f} 秒")
    print(f"事件数：{len(metadata['events'])}")

In [ ]:
# 可视化 EEG 信号
if subjects:
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))
    
    # 显示前 4 个通道的信号
    sample_rate = metadata['sample_rate']
    time = np.arange(eeg_data.shape[1]) / sample_rate
    
    for i, ax in enumerate(axes):
        channel_name = metadata['channel_names'][i]
        ax.plot(time, eeg_data[i] + i * 50)  # 添加偏移便于观察
        ax.set_title(f'Channel {channel_name}')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Amplitude (μV)')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 功率谱分析
if subjects:
    from scipy import signal
    
    processor = EEGProcessor(EEGConfig(sample_rate=sample_rate))
    
    # 预处理
    cleaned = processor.preprocess(eeg_data)
    
    # 计算功率谱
    freqs, psd = processor.calc_power_spectrum(cleaned)
    
    # 可视化
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # 绘制所有通道的平均 PSD
    mean_psd = np.mean(psd, axis=0)
    ax.semilogy(freqs, mean_psd, label='Mean PSD')
    
    # 标注 EEG 频段
    bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13), 
             'beta': (13, 30), 'gamma': (30, 45)}
    
    for band, (low, high) in bands.items():
        ax.axvspan(low, high, alpha=0.2, label=band)
    
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power Spectral Density')
    ax.set_title('Average Power Spectrum')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 50])
    
    plt.tight_layout()
    plt.show()

## 2. DEAP 数据集探索（情绪）

In [ ]:
# DEAP 数据集配置
DEAP_DIR = '/path/to/deap'

deap_loader = DEAPLoader(DEAP_DIR)
deap_info = deap_loader.get_info()

print(f"数据集：{deap_info['name']}")
print(f"被试数量：{deap_info['n_subjects']}")
print(f"通道数：{deap_info['n_channels']}")
print(f"Trial 数量：{deap_info['n_trials']}")
print(f"Trial 时长：{deap_info['trial_duration']} 秒")

In [ ]:
# 加载 DEAP 数据
deap_subjects = deap_loader.get_subjects()

if deap_subjects:
    subject_id = deap_subjects[0]
    eeg_data, metadata = deap_loader.load_subject(subject_id)
    
    print(f"\n被试 {subject_id} 数据:")
    print(f"数据形状：{eeg_data.shape}")
    print(f"(trials, channels, samples)")
    
    # 显示标注信息
    if metadata['labels']:
        print(f"\n标注维度：{metadata['labels'].keys()}")

In [ ]:
# 可视化不同情绪的 EEG 模式
if deap_subjects:
    # 选择一个 trial
    trial_idx = 5
    trial_data = eeg_data[trial_idx]  # (channels, samples)
    
    # 获取该 trial 的情绪标注
    if metadata['labels']:
        labels = metadata['labels']
        valence = labels['labels'][trial_idx, 0]  # 1-9
        arousal = labels['labels'][trial_idx, 1]  # 1-9
        
        print(f"Trial {trial_idx}: Valence={valence}, Arousal={arousal}")
    
    # 可视化
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    
    # 时域信号
    time = np.arange(trial_data.shape[1]) / metadata['sample_rate']
    axes[0].plot(time, trial_data[0])  # Fp1 通道
    axes[0].set_title(f'Trial {trial_idx} - Fp1 Channel')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude (μV)')
    axes[0].grid(True, alpha=0.3)
    
    # 频域分析
    processor = EEGProcessor(EEGConfig(sample_rate=metadata['sample_rate']))
    freqs, psd = processor.calc_power_spectrum(trial_data)
    
    mean_psd = np.mean(psd, axis=0)
    axes[1].semilogy(freqs, mean_psd)
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel('PSD')
    axes[1].set_title('Power Spectrum')
    axes[1].set_xlim([0, 50])
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 3. SEED 数据集探索（情绪）

In [ ]:
# SEED 数据集配置
SEED_DIR = '/path/to/seed'

seed_loader = SEEDLoader(SEED_DIR)
seed_info = seed_loader.get_info()

print(f"数据集：{seed_info['name']}")
print(f"被试数量：{seed_info['n_subjects']}")
print(f"通道数：{seed_info['n_channels']}")
print(f"Sessions: {seed_info['n_sessions']}")
print(f"情绪类别：{seed_info['emotion_classes']}")

In [ ]:
# 加载 SEED 数据
seed_subjects = seed_loader.get_subjects()

if seed_subjects:
    subject_id = seed_subjects[0]
    eeg_data, metadata = seed_loader.load_subject(subject_id, session=1)
    
    print(f"\n被试 {subject_id}, Session 1 数据:")
    print(f"数据形状：{eeg_data.shape}")
    print(f"Trial 数量：{metadata['n_trials']}")
    
    if metadata['labels'] is not None:
        print(f"标注形状：{metadata['labels'].shape}")
        print(f"情绪类别分布：{np.bincount(metadata['labels'])}")

## 4. 特征提取与比较

In [ ]:
# 比较不同数据集的特征
from app.eeg_processor_v2 import generate_synthetic_eeg

processor = EEGProcessor(EEGConfig(sample_rate=256))

# 生成不同专注度的模拟数据
print("=== 不同专注度水平的 EEG 特征比较 ===\n")

attention_levels = [0.2, 0.5, 0.8]
results = []

for level in attention_levels:
    # 生成数据
    data = generate_synthetic_eeg(duration=30, attention_level=level)
    
    # 预处理
    cleaned = processor.preprocess(data)
    
    # 分析
    result = processor.analyze(cleaned)
    results.append(result)
    
    print(f"专注度水平：{level:.1f}")
    print(f"  - Attention Score: {result['attention_score']:.1f}")
    print(f"  - Relaxation Score: {result['relaxation_score']:.1f}")
    print(f"  - Cognitive Load: {result['cognitive_load']:.1f}")
    print()

In [ ]:
# 可视化特征比较
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['attention_score', 'relaxation_score', 'cognitive_load']
titles = ['Attention Score', 'Relaxation Score', 'Cognitive Load']

for ax, metric, title in zip(axes, metrics, titles):
    values = [r[metric] for r in results]
    ax.bar([f'{l:.1f}' for l in attention_levels], values, color=['red', 'green', 'blue'])
    ax.set_title(title)
    ax.set_xlabel('Attention Level')
    ax.set_ylabel('Score')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 数据预处理管道测试

In [ ]:
# 测试不同的伪迹去除方法
if subjects:
    from app.eeg_processor_v2 import generate_synthetic_eeg
    
    # 生成带伪迹的测试数据
    data = generate_synthetic_eeg(duration=30)
    
    # 添加人工伪迹
    t = np.arange(data.shape[1]) / 256
    eog = np.sin(2 * np.pi * 2 * t) * 150  # 眼电伪迹
    data[0, :] += eog
    data[1, :] += eog
    
    processor = EEGProcessor()
    
    # 测试不同方法
    methods = ['threshold', 'ica']
    
    fig, axes = plt.subplots(len(methods) + 1, 1, figsize=(16, 12))
    
    # 原始数据
    axes[0].plot(t, data[0], label='Original')
    axes[0].set_title('Original Data (with EOG artifact)')
    axes[0].set_ylabel('Amplitude (μV)')
    axes[0].grid(True, alpha=0.3)
    
    for i, method in enumerate(methods, 1):
        cleaned = processor.remove_artifacts(data.copy(), method=method)
        axes[i].plot(t, cleaned[0], label=f'{method.upper()}')
        axes[i].set_title(f'After {method.upper()} Artifact Removal')
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Amplitude (μV)')
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. 导出数据集统计报告

In [ ]:
# 生成数据集统计报告
report = {
    'datasets': {
        'physionet': loader.get_info() if subjects else 'Not loaded',
        'deap': deap_loader.get_info() if deap_subjects else 'Not loaded',
        'seed': seed_loader.get_info() if seed_subjects else 'Not loaded',
    },
    'analysis_timestamp': pd.Timestamp.now().isoformat(),
}

import json
with open('../docs/dataset_statistics.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print("数据集统计报告已导出到 ../docs/dataset_statistics.json")

## 总结

本 Notebook 展示了：
1. ✅ PhysioNet 运动想象数据集的加载与可视化
2. ✅ DEAP 情绪数据集的探索
3. ✅ SEED 情绪数据集的分析
4. ✅ EEG 特征提取与比较
5. ✅ 伪迹去除方法测试
6. ✅ 数据集统计报告导出

### 下一步
- 使用真实数据进行模型训练
- 实现跨数据集的迁移学习
- 构建标准化的数据预处理管道